# RibFrac（Zenodo）→ Google Drive

在 Colab 中挂载 Google Drive，将 RibFrac 训练（两部分）、验证与测试图像下载到 **Drive 根目录下** `dataset/pretrain/RibFrac/download/`。

数据分属多条 Zenodo 记录：训练第一部分 [3893508](https://zenodo.org/records/3893508)、训练第二部分 [3893498](https://zenodo.org/records/3893498)、验证 [3893496](https://zenodo.org/records/3893496)、测试仅图像（无单独 info/labels）[3993380](https://zenodo.org/records/3993380)。

下载使用 `wget -c`，支持断点续传（中断后重新运行同一格即可继续）。每个文件会打印 `[序号/总数]` 与进度条；若某链接失效，跑完后会在汇总里列出失败项及 URL。

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import subprocess

DEST = "/content/drive/MyDrive/dataset/pretrain/RibFrac/download"
os.makedirs(DEST, exist_ok=True)

# --show-progress：在 Colab/非 TTY 下也显示单行进度；与 -c 配合可断点续传
WGET_CMD = ["wget", "-c", "--show-progress"]

URLS = [
    (
        "https://zenodo.org/records/3893508/files/ribfrac-train-images-1.zip?download=1",
        "ribfrac-train-images-1.zip",
    ),
    (
        "https://zenodo.org/records/3893508/files/ribfrac-train-info-1.csv?download=1",
        "ribfrac-train-info-1.csv",
    ),
    (
        "https://zenodo.org/records/3893508/files/ribfrac-train-labels-1.zip?download=1",
        "ribfrac-train-labels-1.zip",
    ),
    (
        "https://zenodo.org/records/3893498/files/ribfrac-train-images-2.zip?download=1",
        "ribfrac-train-images-2.zip",
    ),
    (
        "https://zenodo.org/records/3893498/files/ribfrac-train-info-2.csv?download=1",
        "ribfrac-train-info-2.csv",
    ),
    (
        "https://zenodo.org/records/3893498/files/ribfrac-train-labels-2.zip?download=1",
        "ribfrac-train-labels-2.zip",
    ),
    (
        "https://zenodo.org/records/3893496/files/ribfrac-val-images.zip?download=1",
        "ribfrac-val-images.zip",
    ),
    (
        "https://zenodo.org/records/3893496/files/ribfrac-val-info.csv?download=1",
        "ribfrac-val-info.csv",
    ),
    (
        "https://zenodo.org/records/3893496/files/ribfrac-val-labels.zip?download=1",
        "ribfrac-val-labels.zip",
    ),
    (
        "https://zenodo.org/records/3993380/files/ribfrac-test-images.zip?download=1",
        "ribfrac-test-images.zip",
    ),
]

n = len(URLS)
failed = []

for i, (url, filename) in enumerate(URLS, start=1):
    out_path = os.path.join(DEST, filename)
    print(f"\n[{i}/{n}] {filename}")
    print(f"    -> {out_path}")
    print(f"    {url}")
    try:
        subprocess.run([*WGET_CMD, "-O", out_path, url], check=True)
        print(f"    OK")
    except subprocess.CalledProcessError as e:
        failed.append((filename, url, e.returncode))
        print(f"    FAILED (exit {e.returncode})")

print("\n" + "=" * 60)
if failed:
    print(f"下载失败 {len(failed)} 个（共 {n} 个），请对照 URL 检查链接或网络：")
    for filename, url, code in failed:
        print(f"  - {filename}  exit={code}")
        print(f"    {url}")
    raise RuntimeError(f"{len(failed)} 个文件下载失败，见上方列表")
print(f"全部完成（{n} 个）。目录内容：")
print(sorted(os.listdir(DEST)))


[1/10] ribfrac-train-images-1.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/download/ribfrac-train-images-1.zip
    https://zenodo.org/records/3893508/files/ribfrac-train-images-1.zip?download=1
    OK

[2/10] ribfrac-train-info-1.csv
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/download/ribfrac-train-info-1.csv
    https://zenodo.org/records/3893508/files/ribfrac-train-info-1.csv?download=1
    OK

[3/10] ribfrac-train-labels-1.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/download/ribfrac-train-labels-1.zip
    https://zenodo.org/records/3893508/files/ribfrac-train-labels-1.zip?download=1
    OK

[4/10] ribfrac-train-images-2.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/download/ribfrac-train-images-2.zip
    https://zenodo.org/records/3893498/files/ribfrac-train-images-2.zip?download=1
    OK

[5/10] ribfrac-train-info-2.csv
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/download/ribfrac-train-info-2.csv
    https://zenodo.o

In [3]:
import os
import zipfile

DOWNLOAD = "/content/drive/MyDrive/dataset/pretrain/RibFrac/download"
UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/RibFrac/unzip"
os.makedirs(UNZIP_ROOT, exist_ok=True)

zip_names = sorted(f for f in os.listdir(DOWNLOAD) if f.lower().endswith(".zip"))

failed = []
for i, zname in enumerate(zip_names, start=1):
    zpath = os.path.join(DOWNLOAD, zname)
    out_dir = os.path.join(UNZIP_ROOT, os.path.splitext(zname)[0])
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n[{i}/{len(zip_names)}] {zname}")
    print(f"    -> {out_dir}")
    try:
        with zipfile.ZipFile(zpath, "r") as zf:
            bad = zf.testzip()
            if bad is not None:
                raise zipfile.BadZipFile(f"损坏条目: {bad}")
            zf.extractall(out_dir)
        print("    OK")
    except Exception as e:
        failed.append((zname, str(e)))
        print(f"    FAILED: {e}")

print("\n" + "=" * 60)
if failed:
    print(f"解压失败 {len(failed)} 个：")
    for zname, err in failed:
        print(f"  - {zname}: {err}")
    raise RuntimeError("部分 zip 解压失败，见上方列表")
print("全部 zip 解压完成。")
print("unzip 下目录：", sorted(os.listdir(UNZIP_ROOT)))


[1/7] ribfrac-test-images.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/unzip/ribfrac-test-images
    OK

[2/7] ribfrac-train-images-1.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/unzip/ribfrac-train-images-1
    OK

[3/7] ribfrac-train-images-2.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/unzip/ribfrac-train-images-2
    OK

[4/7] ribfrac-train-labels-1.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/unzip/ribfrac-train-labels-1
    OK

[5/7] ribfrac-train-labels-2.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/unzip/ribfrac-train-labels-2
    OK

[6/7] ribfrac-val-images.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/unzip/ribfrac-val-images
    OK

[7/7] ribfrac-val-labels.zip
    -> /content/drive/MyDrive/dataset/pretrain/RibFrac/unzip/ribfrac-val-labels
    OK

全部 zip 解压完成。
unzip 下目录： ['ribfrac-test-images', 'ribfrac-train-images-1', 'ribfrac-train-images-2', 'ribfrac-train-labels-1', 'ribfrac-train-label

In [3]:
import os
from collections import Counter

UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/RibFrac/unzip"

def is_nii_gz(name: str) -> bool:
    return name.lower().endswith(".nii.gz")

def summarize_one_dataset(name: str, root: str):
    n_nii = 0
    n_other = 0
    other_ext = Counter()
    samples = []

    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if is_nii_gz(fn):
                n_nii += 1
                if len(samples) < 5:
                    samples.append(os.path.relpath(os.path.join(dirpath, fn), root))
            else:
                n_other += 1
                stem, ext = os.path.splitext(fn)
                if ext.lower() == ".gz" and stem.lower().endswith(".nii"):
                    # 极少数命名若不走 .nii.gz 后缀，仍算作 nii
                    n_nii += 1
                    n_other -= 1
                    if len(samples) < 5:
                        samples.append(os.path.relpath(os.path.join(dirpath, fn), root))
                else:
                    key = ext.lower() if ext else "(无扩展名)"
                    other_ext[key] += 1

    return n_nii, n_other, other_ext, samples

def print_tree(path: str, prefix: str = "", max_depth: int = 4, max_items: int = 20):
    """深度与每节点展示条数有限，避免医学数据成千上万行刷屏。"""
    rel = os.path.relpath(path, UNZIP_ROOT)
    name = os.path.basename(path.rstrip(os.sep)) if rel != "." else os.path.basename(UNZIP_ROOT.rstrip(os.sep))
    if prefix == "":
        print(f"{name}/")

    if max_depth <= 0:
        print(f"{prefix}  … (已截断深度)")
        return

    try:
        entries = sorted(os.listdir(path))
    except OSError as e:
        print(f"{prefix}  [无法列出: {e}]")
        return

    dirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    files = [e for e in entries if not os.path.isdir(os.path.join(path, e))]
    # 优先展示 .nii.gz，其余按名字排
    nii_files = sorted(f for f in files if is_nii_gz(f))
    rest_files = sorted(f for f in files if not is_nii_gz(f))
    ordered_files = nii_files + rest_files

    shown = 0
    # 先目录
    for d in dirs:
        if shown >= max_items:
            break
        print(f"{prefix}  ├─ {d}/")
        print_tree(
            os.path.join(path, d),
            prefix + "  │   ",
            max_depth=max_depth - 1,
            max_items=max_items,
        )
        shown += 1

    if len(dirs) > shown:
        print(f"{prefix}  ├─ … 还有目录 {len(dirs) - shown} 个未展开")

    # 再文件（本层只列文件名，不递归进子目录里的文件）
    file_shown = 0
    for fn in ordered_files:
        if file_shown >= max(0, max_items - shown):
            break
        tag = " [nii.gz]" if is_nii_gz(fn) else ""
        print(f"{prefix}  ├─ {fn}{tag}")
        file_shown += 1
    if len(ordered_files) > file_shown:
        print(f"{prefix}  ├─ … 本目录还有文件 {len(ordered_files) - file_shown} 个未列出")


print("=" * 72)
print("RibFrac 解压目录概览（以 .nii.gz 为主）")
print("根路径:", UNZIP_ROOT)
print("=" * 72)

subs = sorted(
    d for d in os.listdir(UNZIP_ROOT)
    if os.path.isdir(os.path.join(UNZIP_ROOT, d))
)

if not subs:
    print("（unzip 下没有子目录，请检查路径是否已解压）")
else:
    print("\n【1】各子集统计（每个 zip 对应的一级文件夹）\n")
    for name in subs:
        root = os.path.join(UNZIP_ROOT, name)
        n_nii, n_other, other_ext, samples = summarize_one_dataset(name, root)
        print(f"■ {name}/")
        print(f"    .nii.gz 文件数: {n_nii}")
        print(f"    其他文件数:    {n_other}")
        if other_ext:
            top = ", ".join(f"{ext}:{c}" for ext, c in other_ext.most_common(8))
            print(f"    其他扩展名:    {top}")
        if samples:
            print("    样例（相对该子集根路径）:")
            for s in samples:
                print(f"      - {s}")
        print()

    print("【2】目录树（每个顶层子集下最多向下展 4 层；每节点最多列约 20 项，防刷屏）\n")
    for name in subs:
        print("-" * 72)
        print_tree(os.path.join(UNZIP_ROOT, name), max_depth=4, max_items=20)
        print()

print("完成。若 .nii.gz 数量极大，树里会用「…」省略；统计里的数字仍是全量遍历结果。")

RibFrac 解压目录概览（以 .nii.gz 为主）
根路径: /content/drive/MyDrive/dataset/pretrain/RibFrac/unzip

【1】各子集统计（每个 zip 对应的一级文件夹）

■ ribfrac-test-images/
    .nii.gz 文件数: 160
    其他文件数:    0
    样例（相对该子集根路径）:
      - ribfrac-test-images/RibFrac628-image.nii.gz
      - ribfrac-test-images/RibFrac627-image.nii.gz
      - ribfrac-test-images/RibFrac626-image.nii.gz
      - ribfrac-test-images/RibFrac625-image.nii.gz
      - ribfrac-test-images/RibFrac624-image.nii.gz

■ ribfrac-train-images-1/
    .nii.gz 文件数: 300
    其他文件数:    0
    样例（相对该子集根路径）:
      - Part1/RibFrac128-image.nii.gz
      - Part1/RibFrac127-image.nii.gz
      - Part1/RibFrac126-image.nii.gz
      - Part1/RibFrac125-image.nii.gz
      - Part1/RibFrac124-image.nii.gz

■ ribfrac-train-images-2/
    .nii.gz 文件数: 120
    其他文件数:    0
    样例（相对该子集根路径）:
      - Part2/RibFrac420-image.nii.gz
      - Part2/RibFrac419-image.nii.gz
      - Part2/RibFrac418-image.nii.gz
      - Part2/RibFrac417-image.nii.gz
      - Part2/RibFrac416-image.nii.gz

■